# Export Pipeline

Extract session data from SQL Race and export to common file formats: CSV, Parquet, and HDF5.

**Prerequisites:** `pip install pyarrow tables` for Parquet and HDF5 support.

In [ ]:
import sys
sys.path.insert(0, '.')
from sqlrace_helpers import (
    init_sqlrace, load_session, session_summary,
    list_parameters, extract_parameters
)
from pathlib import Path
import pandas as pd

In [ ]:
SESSION_GUID = "<REPLACE WITH YOUR SESSION GUID>"
OUTPUT_DIR = Path("exports")
OUTPUT_DIR.mkdir(exist_ok=True)

sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID)
session_summary(session)

## Extract all parameters

In [ ]:
params = list_parameters(session)
print(f"Extracting {len(params)} parameter(s)...")

df = extract_parameters(session, params)
print(f"DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())

## Export to CSV

CSV is universally readable. The timestamp index is in nanoseconds.

In [ ]:
csv_path = OUTPUT_DIR / "session_data.csv"
df.to_csv(csv_path)
size_mb = csv_path.stat().st_size / 1e6
print(f"Saved: {csv_path} ({size_mb:.2f} MB)")

## Export to Parquet

Parquet is a columnar format — much smaller than CSV and faster to load. Requires `pyarrow`.

In [ ]:
try:
    parquet_path = OUTPUT_DIR / "session_data.parquet"
    df.to_parquet(parquet_path)
    size_mb = parquet_path.stat().st_size / 1e6
    print(f"Saved: {parquet_path} ({size_mb:.2f} MB)")
except ImportError:
    print("Install pyarrow for Parquet support: pip install pyarrow")

## Export to HDF5

HDF5 supports hierarchical data and is widely used in scientific computing. Requires `tables`.

In [ ]:
try:
    hdf_path = OUTPUT_DIR / "session_data.h5"
    df.to_hdf(hdf_path, key="session", mode="w")
    size_mb = hdf_path.stat().st_size / 1e6
    print(f"Saved: {hdf_path} ({size_mb:.2f} MB)")
except ImportError:
    print("Install tables for HDF5 support: pip install tables")

## Export summary

In [ ]:
print("\nExport complete.")
print(f"  Parameters: {len(params)}")
print(f"  Rows:       {len(df)}")
print(f"  Output dir: {OUTPUT_DIR.resolve()}")

for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name}: {f.stat().st_size / 1e6:.2f} MB")

In [ ]:
client_session.Dispose()
print("Session closed.")